<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/3_clase_fastq_y_control_de_calidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase de Secuenciación NGS: Archivos FASTQ y Control de Calidad

## 1. Fundamento Teórico

### ¿Qué es un archivo FASTQ?
Un archivo **FASTQ** es el formato de texto estándar en bioinformática para almacenar la información de secuencias de nucleótidos (ADN o ARN) generadas por plataformas de secuenciación de alto rendimiento (NGS), junto con sus correspondientes evaluaciones de calidad por base.

A diferencia del formato **FASTA** (que solo almacena el encabezado y la secuencia), el **FASTQ** incluye una métrica crítica: el **Phred Quality Score ($Q$)**.

---

### Anatomía de un Archivo FASTQ
Cada lectura (*read*) en un archivo FASTQ consta exactamente de **4 líneas**:

1. **Línea 1 (`@`)**: Identificador único de la lectura, detalles del secuenciador y coordenadas dentro del *flowcell*.
2. **Línea 2**: La secuencia biológica leída (`A`, `C`, `G`, `T` o `N` para bases indeterminadas).
3. **Línea 3 (`+`)**: Separador sintáctico (a veces repite el identificador de la línea 1).
4. **Línea 4**: Cadena de caracteres **ASCII** de igual longitud que la secuencia, que representa la calidad Phred ($Q$) de cada base.

**Ejemplo de una lectura:**
```text
@SRR1234567.1 1 length=150
AGCTTTTCATTCTGACTGCAACGGGCAATATGTCT
+
FJJJJFJJJJJJJJJJJJFJJJJJFFFJJJJF...
```

---

### Phred Quality Score ($Q$)
El valor Phred indica la probabilidad ($P$) de que una base haya sido llamada incorrectamente por el secuenciador:

$$Q = -10 \log_{10}(P)$$

| Phred Score ($Q$) | Probabilidad de Error | Precisión de la Base | Significado Práctico |
| :--- | :--- | :--- | :--- |
| **Q10** | 1 en 10 ($10\%$) | **$90\%$** | Muy baja calidad. Se descarta. |
| **Q20** | 1 en 100 ($1\%$) | **$99\%$** | Calidad aceptable estándar. |
| **Q30** | 1 en 1,000 ($0.1\%$) | **$99.9\%$** | Estándar de oro en genómica. |
| **Q40** | 1 en 10,000 ($0.01\%$) | **$99.99\%$** | Excelente precisión. |

#### Codificación ASCII (Phred+33)
Para optimizar el tamaño del archivo, cada puntaje Phred numérico se convierte en un símbolo ASCII sumándole 33:

$$\text{Carácter ASCII} = Q + 33$$

* Ejemplo: Un valor $Q = 30$ equivale al carácter ASCII $63$ (`?`). Un valor $Q = 40$ equivale a $73$ (`I`).

---

### ¿De dónde se obtienen y para qué sirven?
* **Origen en el laboratorio:** Muestra biológica $\rightarrow$ Extracción ADN/ARN $\rightarrow$ Preparación de Librería $\rightarrow$ Secuenciación en Flujo (ej. Illumina) $\rightarrow$ Imágenes `.bcl` $\rightarrow$ Base Calling $\rightarrow$ Archivo `.fastq`.
* **Repositorios públicos:** NCBI SRA (Sequence Read Archive), ENA, DDBJ.
* **Aplicaciones:** Control de calidad, alineamiento a genoma de referencia, ensamble *de novo*, llamado de variantes (SNPs), cuantificación de expresión génica (RNA-Seq).

# 2. Práctica de Laboratorio en Google Colab

### Objetivos de la Práctica:
1. Inspeccionar la estructura física de un archivo FASTQ.
2. Evaluar métricas de calidad iniciales usando **FastQC**.
3. Filtrar y recortar lecturas de baja calidad/adaptadores con **fastp**.
4. Comparar el estado de los datos antes y después de la limpieza.

### Paso 1: Instalación de Herramientas Bioinformáticas

In [ ]:
# Instalación de FastQC y fastp en la máquina virtual de Colab
!apt-get update -qq
!apt-get install -y fastqc fastp

In [ ]:
pip install Biopython

In [ ]:
# Importación de Biopython
from Bio.Seq import Seq
from Bio import SeqIO
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

### Paso 2: Descarga de Datos de Prueba

In [ ]:
# 1. Instalación de librerías en Colab
!pip install -q pysradb requests pandas

In [ ]:
import os
import gzip
import requests
import pandas as pd
from Bio import Entrez

# Configuración de Entrez de NCBI (requiere un correo para la API)
Entrez.email = "tu_correo@ejemplo.com"

# Definir términos de búsqueda agroalimentarios
search_terms = ['maiz']
max_reads = 10000  # Descargar solo 10,000 lecturas para que tome menos de 3 segundos


def get_srr_from_query(term, max_results=3):
    """
    Busca accesiones SRR en NCBI SRA para un término de búsqueda.
    """
    try:
        handle = Entrez.esearch(db="sra", term=term, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()

        id_list = record.get("IdList", [])
        if not id_list:
            return []

        # Obtener los identificadores SRR a partir de los UIDs de SRA
        summary_handle = Entrez.esummary(db="sra", id=",".join(id_list))
        summaries = Entrez.read(summary_handle)
        summary_handle.close()

        srr_list = []
        for sum_item in summaries:
            exp_xml = sum_item.get("Runs", "")
            # Extraer el código SRR del XML de respuesta
            import re
            found_srr = re.findall(r'acc="(SRR\d+|ERR\d+|DRR\d+)"', exp_xml)
            srr_list.extend(found_srr)

        return list(set(srr_list))
    except Exception as e:
        print(f"Error consultando NCBI para '{term}': {e}")
        return []


def download_fastq_sample_from_ena(srr_id, max_reads=10000, output_dir="teaching_data"):
    """
    Descarga directamente las primeras 'max_reads' desde el servidor europeo (ENA).
    No requiere fasterq-dump ni sra-toolkit.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    out_file = os.path.join(output_dir, f"{srr_id}_sample.fastq")

    # Consultar URL del FASTQ en la API de ENA
    ena_url = f"https://www.ebi.ac.uk/ena/portal/api/filereport?accession={srr_id}&result=read_run&fields=fastq_ftp&format=tsv"

    try:
        df_ena = pd.read_csv(ena_url, sep='\t')
        if df_ena.empty or 'fastq_ftp' not in df_ena.columns or pd.isna(df_ena['fastq_ftp'].values[0]):
            print(f"   [!] No se encontró enlace FASTQ directo en ENA para {srr_id}")
            return False

        ftp_path = df_ena['fastq_ftp'].values[0].split(';')[0]
        http_url = f"https://{ftp_path}"

        print(f"   [+] Descargando muestra ligera de {srr_id}...")

        # Streaming de descarga: detenerse al escribir (max_reads * 4) líneas
        response = requests.get(http_url, stream=True)
        lines_written = 0
        target_lines = max_reads * 4

        with gzip.open(response.raw, 'rt') as gz_in, open(out_file, 'w') as txt_out:
            for line in gz_in:
                txt_out.write(line)
                lines_written += 1
                if lines_written >= target_lines:
                    break

        print(f"   [✓] Éxito: {lines_written // 4} lecturas guardadas en '{out_file}'")
        return True

    except Exception as e:
        print(f"   [!] Error procesando {srr_id}: {e}")
        return False


# Flujo principal de ejecución
for term in search_terms:
    print(f"\nBuscando estudios en SRA para: {term}")
    srr_ids = get_srr_from_query(term, max_results=3)

    if srr_ids:
        print(f"Accesiones encontradas: {srr_ids}")
        for srr_id in srr_ids:
            success = download_fastq_sample_from_ena(srr_id, max_reads=max_reads)
            if success:
                break  # Detenerse tras descargar el primer archivo exitoso para la clase
    else:
        print("No se encontraron resultados.")

In [ ]:
import gzip
from collections import defaultdict
import seaborn as sns
import matplotlib.pyplot as plt
from Bio import SeqIO

%matplotlib inline

# Ruta a tu archivo descargado
fastq_path = '/content/teaching_data/ERR13992929_sample.fastq'

# Abrir el archivo FASTQ de texto plano directamente con SeqIO
recs = SeqIO.parse(fastq_path, 'fastq')

# Obtener el primer registro (lectura)
rec = next(recs)

# Inspeccionar la estructura del objeto SeqRecord
print("--- Objeto SeqRecord Completo ---")
print(rec)

print("\n--- Atributos Individuales ---")
print("ID de la lectura:", rec.id)
print("Descripción completa:", rec.description)
print("Secuencia (primeras 50 bases):", rec.seq[:50])

print("\n--- Anotaciones de Calidad (Phred Scores) ---")
# rec.letter_annotations es un diccionario con la clave 'phred_quality'
phred_scores = rec.letter_annotations['phred_quality']
print("Puntajes Phred por base (primeras 20 bases):", phred_scores[:20])

In [ ]:
# Muestrear las primeras 1,000 lecturas para calcular la calidad promedio por posición
recs = SeqIO.parse(fastq_path, 'fastq')
quality_per_position = defaultdict(list)

for i, record in enumerate(recs):
    if i >= 1000: # Limitar a 1000 lecturas para velocidad
        break
    scores = record.letter_annotations['phred_quality']
    for pos, score in enumerate(scores):
        quality_per_position[pos].append(score)

# Preparar datos para graficar con Seaborn
positions = []
mean_qualities = []

for pos in sorted(quality_per_position.keys()):
    scores = quality_per_position[pos]
    positions.append(pos + 1)
    mean_qualities.append(sum(scores) / len(scores))

# Graficar
plt.figure(figsize=(10, 4))
sns.lineplot(x=positions, y=mean_qualities, color='navy', linewidth=2)
plt.axhline(y=30, color='green', linestyle='--', label='Q30 (99.9% precisión)')
plt.axhline(y=20, color='orange', linestyle='--', label='Q20 (99% precisión)')

plt.title('Perfil de Calidad Promedio por Posición de Lectura (Phred Score)')
plt.xlabel('Posición en la Lectura (pb)')
plt.ylabel('Puntaje Phred Promedio ($Q$)')
plt.ylim(0, 42)
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.show()

### Phred Quality Score ($Q$)
El valor Phred indica la probabilidad ($P$) de que una base haya sido llamada incorrectamente por el secuenciador:

$$Q = -10 \log_{10}(P)$$

| Phred Score ($Q$) | Probabilidad de Error | Precisión de la Base | Significado Práctico |
| :--- | :--- | :--- | :--- |
| **Q10** | 1 en 10 ($10\%$) | **$90\%$** | Muy baja calidad. Se descarta. |
| **Q20** | 1 en 100 ($1\%$) | **$99\%$** | Calidad aceptable estándar. |
| **Q30** | 1 en 1,000 ($0.1\%$) | **$99.9\%$** | Estándar de oro en genómica. |
| **Q40** | 1 en 10,000 ($0.01\%$) | **$99.99\%$** | Excelente precisión. |

In [ ]:
from collections import defaultdict
from Bio import SeqIO

# 1. Abrir el archivo FASTQ directamente (texto plano) con SeqIO.parse
recs = SeqIO.parse(fastq_path, 'fastq')

count = defaultdict(int)

# 2. Contar la frecuencia de cada nucleótido
for rec in recs:
    for letter in rec.seq:
        count[letter] += 1

# 3. Calcular el total de bases
tot = sum(count.values())

# 4. Imprimir los resultados (porcentaje y conteo absoluto)
print(f"Total de bases analizadas: {tot:,}\n")
for letter, num_occurrences in sorted(count.items()):
    percentage = 100. * num_occurrences / tot
    print(f"{letter}: {percentage:.2f}% ({num_occurrences:,} bases)")

# 🧬 Explicación del Código Bioinformático y el Significado de 'N'

---

## 1. ¿Qué hace el código original?

Este script procesa un archivo de secuenciación en formato **FASTQ** para calcular el **conteo total y la frecuencia relativa (porcentaje)** de cada nucleótido presente en las lecturas.

### Desglose paso a paso:
1. **Lectura de secuencias:** Utiliza `Bio.SeqIO.parse` de **Biopython** para iterar sobre los registros del archivo FASTQ.
2. **Conteo por nucleótido:** Recorre cada carácter de cada secuencia (`rec.seq`) y actualiza las frecuencias en un diccionario `defaultdict`.
3. **Cálculo global:** Suma todos los conteos para determinar la cantidad total de bases analizadas.
4. **Reporte:** Imprime el total absoluto y el desglose de cada carácter ordenado alfabéticamente en formato: `Porcentaje% (Conteo absoluto)`.

---

## 2. ¿Qué significa el carácter 'N'?

En archivos de secuencias genéticas (FASTQ/FASTA), la **`N`** representa un **nucleótido desconocido o ambiguo** (*aNy base*).

* **Origen:** Se genera cuando el software del secuenciador (p. ej., Illumina, PacBio, Nanopore) no detecta con suficiente claridad la señal fluorescente o eléctrica de una posición y no puede determinar si la base es **A**, **C**, **G** o **T**.
* **Estándar IUPAC:** `N` es el código oficial para *"cualquier nucleótido"*.
* **Control de Calidad (QC):** Un porcentaje elevado de `N` en un archivo indica baja calidad de secuenciación. En flujos bioinformáticos es habitual realizar un filtrado o *trimming* (recorte) para eliminar secuencias con alto contenido de `N`.

---

## 3. Ejemplo de Salida del Script

```text
Total de bases analizadas: 1,500,000

A: 28.50% (427,500 bases)
C: 21.30% (319,500 bases)
G: 21.70% (325,500 bases)
N: 0.10% (1,500 bases)
T: 28.40% (426,000 bases)

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
from Bio import SeqIO

# 1. Usar SeqIO directamente con el archivo de texto plano
recs = SeqIO.parse('/content/teaching_data/ERR13992929_sample.fastq', 'fastq')

n_cnt = defaultdict(int)
max_len = 0

# 2. Recorrer las lecturas y contar 'N' por posición
for rec in recs:
    seq_len = len(rec.seq)
    if seq_len > max_len:
        max_len = seq_len

    for i, letter in enumerate(rec.seq):
        if letter == 'N':
            n_cnt[i + 1] += 1

# 3. Generar el gráfico de forma segura (incluso si no hay letras 'N')
if max_len > 0:
    positions = list(range(1, max_len + 1))
    n_counts = [n_cnt[pos] for pos in positions]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(positions, n_counts, color='crimson', linewidth=2, marker='o', markersize=3)
    ax.set_xlim(1, max_len)
    ax.set_title("Frecuencia de Bases Indeterminadas ('N') por Posición de Lectura", fontsize=14)
    ax.set_xlabel("Posición en la Lectura (pb)", fontsize=12)
    ax.set_ylabel("Cantidad de 'N' Detectadas", fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.show()

    total_n = sum(n_cnt.values())
    print(f"Total de bases 'N' encontradas en el archivo: {total_n}")
else:
    print("No se encontraron lecturas en el archivo.")

# 📊 Análisis de Distribución Posicional de Bases Indeterminadas ('N') en Lecturas FASTQ

---

## 💡 ¿Qué hace este código?

Este script procesa un archivo de secuenciación en formato **FASTQ** para **medir y graficar en qué posiciones específicas de las lecturas aparecen las bases indeterminadas (`N`)**.

Su objetivo principal en bioinformática es evaluar el **perfil de calidad posicional** de una corrida de secuenciación, lo que permite identificar si los errores o ambigüedades se concentran en zonas específicas (por ejemplo, al final de las lecturas).

---

## 🔍 Explicación Paso a Paso

### 1. **Carga y Lectura del Archivo FASTQ**
* Se utiliza `Bio.SeqIO.parse()` de **Biopython** para leer el archivo `/content/teaching_data/ERR13992929_sample.fastq`.
* Se inicializan:
  * `n_cnt`: Un diccionario predeterminado (`defaultdict(int)`) para registrar cuántas veces aparece una `N` en cada posición (índice 1-based).
  * `max_len`: Una variable para calcular la longitud máxima de las lecturas procesadas.

---

### 2. **Iteración sobre las Lecturas y Conteo Posicional de 'N'**
* Recorre cada lectura (`rec`):
  * Mantiene actualizado `max_len` según el tamaño de la secuencia actual (`len(rec.seq)`).
  * Con `enumerate(rec.seq)`, recorre cada base junto con su índice `i` (0, 1, 2...).
  * Si la base es igual a `'N'`, incrementa en 1 el conteo para esa posición específica: `n_cnt[i + 1]`.

---

### 3. **Construcción y Visualización del Gráfico con Matplotlib**
Si el archivo contiene lecturas (`max_len > 0`):
1. **Preparación de Datos:**
   * Crea una lista de posiciones de `1` a `max_len`.
   * Construye la lista `n_counts` obteniendo el total de `N` para cada posición desde el diccionario `n_cnt`.
2. **Generación de la Gráfica (`plt.subplots`):**
   * Dibuja una línea de color carmesí (`crimson`) con puntos (`marker='o'`) que muestra en el **eje X** la posición dentro de la lectura (pares de bases, *pb*) y en el **eje Y** la frecuencia de `N`.
   * Añade títulos, etiquetas para los ejes y una cuadrícula estilo marca de agua (`grid`).
3. **Resumen Final:**
   * Muestra la gráfica en pantalla (`plt.show()`).
   * Imprime la suma global de todas las bases `'N'` detectadas en el archivo.

---

## 📈 ¿Cómo interpretar los resultados?

* **Picos al final de la lectura:** Es el patrón más común en tecnologías como **Illumina**. La calidad de secuenciación suele decayendo a medida que aumenta la longitud de la lectura, incrementando la presencia de `N`.
* **Picos al inicio o patrones anómalos:** Pueden indicar problemas mecánicos, de reactivos o ruido óptico en los primeros ciclos de secuenciación.

In [ ]:
recs = SeqIO.parse('/content/teaching_data/ERR13992929_sample.fastq', 'fastq')
cnt_qual = defaultdict(int)
for rec in recs:
    for i, qual in enumerate(rec.letter_annotations['phred_quality']):
        if i < 25:
            continue
        cnt_qual[qual] += 1
tot = sum(cnt_qual.values())
for qual, cnt in cnt_qual.items():
    print('%d: %.2f %d' % (qual, 100. * cnt / tot, cnt))

In [ ]:
recs = SeqIO.parse('/content/teaching_data/ERR13992929_sample.fastq', 'fastq')
qual_pos = defaultdict(list)
for rec in recs:
    for i, qual in enumerate(rec.letter_annotations['phred_quality']):
        if i < 25 or qual == 40:
            continue
        pos = i + 1
        qual_pos[pos].append(qual)
vps = []
poses = list(qual_pos.keys())
poses.sort()
for pos in poses:
    vps.append(qual_pos[pos])
fig, ax = plt.subplots(figsize=(16,9))
sns.boxplot(data=vps, ax=ax)
ax.set_xticklabels([str(x) for x in range(26, max(qual_pos.keys()) + 1)])
pass

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from Bio import SeqIO

# 1. Leer todas las posiciones y todos los puntajes Phred sin omitir valores
recs = SeqIO.parse('/content/teaching_data/ERR13992929_sample.fastq', 'fastq')

data = []
for record in recs:
    scores = record.letter_annotations['phred_quality']
    for pos, qual in enumerate(scores):
        data.append({'Posicion': pos + 1, 'Phred': qual})

df = pd.DataFrame(data)

# 2. Calcular métricas resumidas por posición
df_summary = df.groupby('Posicion')['Phred'].agg(
    Promedio='mean',
    Mediana='median',
    Q25=lambda x: x.quantile(0.25),
    Q75=lambda x: x.quantile(0.75)
).reset_index()

# 3. Graficar estilo FastQC
fig, ax = plt.subplots(figsize=(14, 6))

# Zonas de calidad FastQC
ax.axhspan(28, 42, color='#e6ffe6', alpha=0.8, zorder=1) # Verde: Buena calidad
ax.axhspan(20, 28, color='#ffffcc', alpha=0.8, zorder=1) # Amarillo: Calidad aceptable
ax.axhspan(0, 20,  color='#ffe6e6', alpha=0.8, zorder=1) # Rojo: Mala calidad

# Rango Intercuartílico (IQR) y Líneas de Mediana/Promedio
ax.fill_between(df_summary['Posicion'], df_summary['Q25'], df_summary['Q75'],
                color='#3385ff', alpha=0.3, label='Rango Intercuartílico (Q25 - Q75)', zorder=2)
ax.plot(df_summary['Posicion'], df_summary['Mediana'], color='blue', linewidth=2, label='Mediana', zorder=3)
ax.plot(df_summary['Posicion'], df_summary['Promedio'], color='darkred', linestyle='--', linewidth=1.5, label='Promedio', zorder=3)

# Líneas de umbral
ax.axhline(30, color='green', linestyle=':', label='Q30 (99.9% precisión)')
ax.axhline(20, color='orange', linestyle=':', label='Q20 (99% precisión)')

# Formato de ejes
ax.set_title('Perfil de Calidad por Posición de Lectura (Estilo FastQC)', fontsize=14, fontweight='bold')
ax.set_xlabel('Posición en la Lectura (pb)', fontsize=12)
ax.set_ylabel('Puntaje de Calidad Phred ($Q$)', fontsize=12)
ax.set_ylim(0, 42)
ax.set_xlim(1, df_summary['Posicion'].max())
ax.grid(True, linestyle='--', alpha=0.4, zorder=1)
ax.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

### Resumen Diagnóstico del Control de Calidad (Estilo FastQC)

| Métrica / Elemento | Valor / Comportamiento en el Gráfico | Interpretación Bioinformática | Acción Recomendada |
| :--- | :--- | :--- | :--- |
| **Zona Verde ($Q \ge 28$)** | Ocupada durante casi todo el recorrido ($1$ - $240$ pb)[cite: 2] | **Calidad Alta:** Precisión $> 99.85\%$ por base en casi toda la secuencia[cite: 2]. | Ninguna acción requerida[cite: 2]. |
| **Zona Amarilla ($20 \le Q < 28$)** | Solo dos caídas puntuales de la dispersión ($Q25$)[cite: 2] | **Calidad Aceptable:** Errores entre $0.1\%$ y $1\%$ en un porcentaje muy reducido de lecturas[cite: 2]. | Monitorear el extremo $3'$[cite: 2]. |
| **Zona Roja ($Q < 20$)** | Ninguna tendencia global entra en esta zona[cite: 2] | **Mala Calidad:** Ausencia de regiones con tasa de error $> 1\%$[cite: 2]. | No requiere descarte de lecturas completas[cite: 2]. |
| **Mediana (Línea Azul)** | Estable en $Q \approx 37$[cite: 2] | Más del $50\%$ de las lecturas tienen una precisión del $99.98\%$[cite: 2]. | Muestra apta para alineamiento/ensamble[cite: 2]. |
| **Promedio (Línea Roja)** | $Q36 \rightarrow Q33$ (caída suave constante)[cite: 2] | **Degeneración estándar NGS:** Pérdida progresiva de sincronía óptica (*phasing*) hacia el extremo $3'$. | Opcional: *Soft-trimming* de las últimas 15 pb ($> 225$ pb)[cite: 2]. |
| **Dispersión IQR (Sombra Azul)** | Estrecha globalmente; picos hacia abajo en pb $\sim 228$ y $\sim 238$[cite: 2] | Ruido o baja señal momentánea en ciclos específicos del *flowcell*[cite: 2]. | Filtrado suave con `fastp` (`-q 20 -l 50`)[cite: 2]. |

### Paso 3: Inspección Estructural del ARCHIVO FASTQ

In [ ]:
fastq_path = '/content/teaching_data/ERR13992929_sample.fastq'

# El prefijo $ le indica a Bash que use la variable de Python
!head -n 16 $fastq_path

In [ ]:
# Contar el número total de lecturas en la muestra
fastq_path = '/content/teaching_data/ERR13992929_sample.fastq'

# Usamos $fastq_path para que Bash evalúe la variable de Python
!echo -n "Número total de lecturas crudas: "
!expr $(wc -l < $fastq_path) / 4

### Paso 4: Control de Calidad Inicial (FastQC)

In [ ]:
import os
from IPython.display import IFrame, display

# 1. Definir la ruta real de tu archivo FASTQ
fastq_path = "/content/teaching_data/ERR13992929_sample.fastq"
output_dir = "/content/qc_reports"

os.makedirs(output_dir, exist_ok=True)

# 2. Instalar FastQC si no está instalado
!apt-get update -qq && apt-get install -y -qq fastqc

# 3. Ejecutar FastQC
!fastqc {fastq_path} -o {output_dir}

# 4. Obtener el nombre del archivo HTML generado automáticamente por FastQC
base_name = os.path.basename(fastq_path).replace(".fastq", "_fastqc.html")
html_path = os.path.join(output_dir, base_name)

# 5. Desplegar el reporte HTML de forma limpia dentro del Notebook usando un iframe
print(f"Desplegando reporte FastQC: {html_path}\n")
display(IFrame(src=f"files{html_path}", width="100%", height=600))

In [ ]:
import os
from IPython.display import HTML, display

# 1. Definir rutas
fastq_path = "/content/teaching_data/ERR13992929_sample.fastq"
output_dir = "/content/qc_reports"

os.makedirs(output_dir, exist_ok=True)

# 2. Instalar FastQC
!apt-get update -qq && apt-get install -y -qq fastqc

# 3. Ejecutar FastQC
!fastqc {fastq_path} -o {output_dir}

# 4. Obtener ruta del archivo generado
base_name = os.path.basename(fastq_path).replace(".fastq", "_fastqc.html")
html_path = os.path.join(output_dir, base_name)

# 5. Cargar y mostrar el contenido HTML directamente
print(f"Desplegando reporte FastQC: {html_path}\n")

with open(html_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

display(HTML(html_content))

In [ ]:
import zipfile

# Ruta al archivo zip generado por FastQC
zip_path = html_path.replace(".html", ".zip")
extract_dir = os.path.join(output_dir, "fastqc_unzipped")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Buscar y leer el archivo summary.txt
folder_inside = os.path.basename(zip_path).replace(".zip", "")
summary_file = os.path.join(extract_dir, folder_inside, "summary.txt")

print("--- DIAGNÓSTICO RÁPIDO DE FASTQC ---")
with open(summary_file, 'r') as f:
    print(f.read())

In [ ]:
import os
import glob
from IPython.display import HTML, display

# 1. Definir la ruta exacta del archivo de entrada
fastq_file = "/content/teaching_data/ERR13992929_sample.fastq"

# Verificar si el archivo FASTQ realmente existe antes de continuar
if not os.path.exists(fastq_file) and not os.path.exists(f"/content/{fastq_file}"):
    print(f"⚠️ El archivo '{fastq_file}' no existe. Revisa la ruta o que se haya descargado correctamente.")
else:
    # 2. Ejecutar FastQC guardando en la carpeta actual
    !fastqc {fastq_file} -o ./

    # 3. Buscar automáticamente cualquier archivo HTML generado por FastQC
    html_files = glob.glob("*_fastqc.html")

    if html_files:
        html_file = html_files[0]
        print(f"✅ Reporte encontrado: {html_file}")

        with open(html_file, "r", encoding="utf-8") as f:
            html_raw = f.read()

        display(HTML(html_raw))
    else:
        print("❌ Ocurrió un error: FastQC no generó ningún archivo HTML. Revisa si el archivo FASTQ está dañado.")

In [ ]:
# 1. Ejecutar FastQC sobre las lecturas crudas
!fastqc ERR13993570_sample.fastq -o ./

# 2. Desplegar el reporte HTML dentro del Notebook
from IPython.display import HTML, display

# Usar el nombre exacto que genera FastQC para la muestra
html_file = "/content/ERR13992929_sample_fastqc.html"

with open(html_file, "r", encoding="utf-8") as f:
    html_raw = f.read()

display(HTML(html_raw))

### Paso 5: Filtrado y Recorte de Calidad con `fastp`

In [ ]:
# Ejecutar fastp para limpiar los datos
!fastp \
  -i /content/teaching_data/ERR13992929_sample.fastq \
  -o muestra_R1_filtrada.fastq \
  -h reporte_fastp.html \
  -j reporte_fastp.json \
  -q 20 \
  -u 30 \
  -l 50

### Paso 6: Evaluación de Resultados Post-Filtrado

In [ ]:
# Mostrar el reporte interactivo generado por fastp
with open("/content/reporte_fastp.html", "r") as f:
    html_fastp = f.read()

display(HTML(html_fastp))

In [ ]:
# Comparación de volumen de lecturas pre y post filtrado
!echo -n "Lecturas originales (Raw):      "
!expr $(wc -l < /content/teaching_data/ERR13992929_sample.fastq) / 4

!echo -n "Lecturas conservadas (Clean):   "
!expr $(wc -l < muestra_R1_filtrada.fastq) / 4

### Paso 7: Confirmación de Calidad Final (FastQC)

In [ ]:
# FastQC sobre la muestra limpia
!fastqc muestra_R1_filtrada.fastq -o ./

with open("muestra_R1_filtrada_fastqc.html", "r") as f:
    html_clean = f.read()

display(HTML(html_clean))

---

## 🧬 Próximos pasos en el análisis bioinformático (Post-filtrado)

Una vez realizado el control de calidad inicial y el filtrado/limpieza de las lecturas (*trimming* y remoción de adaptadores), el flujo de trabajo depende del **objetivo biológico** de nuestro experimento:

---

### 1. Control de Calidad Posterior (Re-QC)
> **Objetivo:** Confirmar que el filtrado eliminó con éxito las bases de baja calidad y los adaptadores.
* **Herramientas:** `FastQC` (sobre los archivos limpios) y `MultiQC` (para comparar los reportes antes y después del filtrado en una sola vista).

---

### 2. Mapeo / Alineamiento (Si existe genoma de referencia)
> **Objetivo:** Determinar la ubicación exacta en el genoma de cada una de nuestras lecturas secuenciadas.
* **ADN / Resecuenciación:** `BWA-MEM` o `Bowtie2`
* **RNA-Seq (Alineación considerando splices/intrones):** `STAR` o `HISAT2`
* **Archivos generados:** SAM $\rightarrow$ BAM (versión binaria indexada).

---

### 3. Cuantificación de Expresión Génica (RNA-Seq rápido)
> **Objetivo:** Medir la abundancia de los transcritos/genes omitiendo el alineamiento tradicional.
* **Herramientas:** `Kallisto` o `Salmon` (pseudo-alineadores).
* **Archivos generados:** Tablas de recuentos de lecturas (*counts*) en `.tsv`.

---

### 4. Ensamblado *De Novo* (Si NO existe genoma de referencia)
> **Objetivo:** Reconstruir el genoma o transcriptoma uniendo las lecturas por solapamiento.
* **Genomas (ADN):** `SPAdes` o `MEGAHIT`
* **Transcriptomas (RNA):** `Trinity`
* **Archivos generados:** Secuencias contiguas en formato FASTA (`.fasta`).

---

### 🗺️ Flujo de Trabajo General (*Pipeline*)

```text
               [ FASTQ Filtrado y Limpio ]
                            │
         ┌──────────────────┴──────────────────┐
         ▼                                     ▼
[ Con Genoma de Referencia ]          [ Sin Genoma de Referencia ]
         │                                     │
   Alineamiento                          Ensamblado De Novo
 (BWA / STAR / HISAT2)                  (SPAdes / Trinity)
         │                                     │
    Archivo BAM                           Contigs / FASTA
         │                                     │
 ┌───────┴───────┐                             ▼
 ▼               ▼                     Anotación Funcional
Llamado de   Cuantificación
Variantes    de Expresión
(GATK)     (featureCounts)